In [1]:
import pandas as pd
df = pd.read_csv('complaints_realistic.csv')
print(df.head())
print(df['category'].value_counts())

                                      complaint_text category
0  Garbage has not been collected from the market...  garbage
1                Broken speed breaker near bus stand    roads
2  Illegal parking next to the overflowing dustbi...    other
3  Construction waste has been dumped near the sc...    roads
4  A burst water pipeline has created a massive p...    water
category
garbage        219
water          191
electricity    187
roads          166
other          123
Name: count, dtype: int64


In [2]:
import re
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df['clean_text'] = df['complaint_text'].apply(clean_text)

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['category'], test_size=0.2, random_state=42)


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=1000, ngram_range=(1,2))),
    ('clf', LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=1000, ngram_range=(1, 2))),
                ('clf', LogisticRegression(max_iter=1000))])

In [5]:
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(accuracy_score(y_test, y_pred))


              precision    recall  f1-score   support

 electricity       0.97      0.97      0.97        35
     garbage       1.00      1.00      1.00        43
       other       1.00      1.00      1.00        24
       roads       1.00      0.97      0.99        36
       water       0.98      1.00      0.99        40

    accuracy                           0.99       178
   macro avg       0.99      0.99      0.99       178
weighted avg       0.99      0.99      0.99       178

0.9887640449438202


In [6]:
import pickle
pickle.dump(model, open("text_model.pkl", "wb"))

In [7]:
model = pickle.load(open("text_model.pkl", "rb"))
print(model.predict(["there is open puddle on the street near electric pole"]))  # output: ['garbage']
print(model.predict(["potholes on road"])) 

['electricity']
['roads']
